# Cause classifier: why did the incident happen?

Predicts PHMSA's apparent cause from the narrative. Same model, encoding and training settings as the severity classifier (`Training Runs.ipynb`), so the two are directly comparable.

1. **Runtime → Change runtime type → T4 GPU** (cell 1 stops if there is no GPU).
2. Upload `cause_colab_export.json` (or `cause_detail_colab_export.json` for sub-causes), made by `python cause_model/export_for_colab.py`.
3. Run all cells. About 10 to 15 minutes on a T4.

With the sub-cause export, cell 5 also rolls predictions up to the 8 parent causes, and cell 6 downloads `phmsa_cause_detail_model.zip` (unzip it to `cause_model/phmsa_cause_detail_model`).

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import json

EXPORT = "cause_colab_export.json"          # or "cause_detail_colab_export.json"
with open(EXPORT) as f:
    data = json.load(f)
LABELS = data["labels"]
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}
print(f"{data['level']} level, {len(LABELS)} labels:", {k: len(data[k]) for k in ("train", "dev", "test")})

In [ ]:
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 512
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(rows):
    return Dataset.from_dict({"text": [r["text"] for r in rows], "labels": [r["label"] for r in rows]})

raw = DatasetDict({k: to_ds(data[k]) for k in ("train", "dev", "test")})

def encode(batch):
    # same as severity: no special tokens first, keep the start and the end of long narratives
    enc = tokenizer(batch["text"], add_special_tokens=False, truncation=False)
    input_ids = []
    for ids in enc["input_ids"]:
        room = MAX_LEN - 2
        if len(ids) > room:
            head = room // 2
            ids = ids[:head] + ids[-(room - head):]
        input_ids.append([tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id])
    return {"input_ids": input_ids, "attention_mask": [[1] * len(x) for x in input_ids]}

tokenized = raw.map(encode, batched=True, remove_columns=["text"])
lens = np.array([len(x) for x in tokenized["train"]["input_ids"]])
print(f"median length {int(np.median(lens))} tokens; {np.mean(lens >= MAX_LEN):.1%} of training narratives were cut")

In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, EarlyStoppingCallback, set_seed)
from sklearn.metrics import f1_score, accuracy_score

set_seed(42)
counts = np.bincount([r["label"] for r in data["train"]], minlength=len(LABELS))
weights = (counts.sum() / (len(LABELS) * counts)) ** 0.5      # square-root inverse frequency, as for severity
class_weights = torch.tensor(weights, dtype=torch.float)
for l, c, w in zip(LABELS, counts, weights):
    print(f"  {l:<45} {c:>5}  weight {w:.2f}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def macro_f1(y_true, y_pred):
    present = sorted(set(y_true))
    return f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"macro_f1": macro_f1(labels, preds), "accuracy": accuracy_score(labels, preds)}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels,
                                                 weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir="./cause_checkpoint", eval_strategy="epoch", save_strategy="epoch",
    learning_rate=3e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=8, weight_decay=0.01, fp16=True, seed=42,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", save_total_limit=2, report_to="none")

trainer = WeightedTrainer(
    model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["dev"],
    processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()

evals = [h for h in trainer.state.log_history if "eval_macro_f1" in h]
best = max(evals, key=lambda h: h["eval_macro_f1"])
print(f"best epoch {int(best['epoch'])}: dev macro-F1 {best['eval_macro_f1']:.3f}, accuracy {best['eval_accuracy']:.3f}")

In [ ]:
# Test set, scored once
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

out = trainer.predict(tokenized["test"])
y_true, y_pred = out.label_ids, out.predictions.argmax(-1)
test_commodity = np.array([r["commodity_type"] for r in data["test"]])
ALL = list(range(len(LABELS)))

print(classification_report(y_true, y_pred, labels=ALL, target_names=LABELS, digits=3, zero_division=0))
print(f"Overall macro-F1 {macro_f1(y_true, y_pred):.3f}   accuracy {accuracy_score(y_true, y_pred):.3f}")
if len(LABELS) <= 12:
    short = [l[:14] for l in LABELS]
    print("\nConfusion matrix (rows = true, columns = predicted):")
    print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=ALL), index=short, columns=short).to_string())

print("\nPer commodity (macro-F1 over classes present):")
for c in sorted(set(test_commodity)):
    m = test_commodity == c
    print(f"  {c:<28} n={int(m.sum()):<5} macro-F1 {macro_f1(y_true[m], y_pred[m]):.3f}   "
          f"accuracy {accuracy_score(y_true[m], y_pred[m]):.3f}")

if data["level"] == "detail":
    # roll sub-causes up to their parent category, to compare with the 8-category model
    parent = lambda i: LABELS[i].split(" - ")[0]
    pt, pp = np.array([parent(i) for i in y_true]), np.array([parent(i) for i in y_pred])
    print(f"\nRolled up to the 8 parent causes: macro-F1 {macro_f1(pt, pp):.3f}   accuracy {np.mean(pt == pp):.3f}"
          "   (the 8-category model: 0.748 / 0.818)")
    print(f"Sub-cause wrong but parent right: {np.mean((y_true != y_pred) & (pt == pp)):.1%} of test narratives")

In [ ]:
# Save and download (cause_model/check_cause_local.py should reproduce the numbers above)
OUT = "phmsa_cause_model" if data["level"] == "cause" else "phmsa_cause_detail_model"
trainer.save_model(f"./{OUT}")
tokenizer.save_pretrained(f"./{OUT}")
import os, shutil
if os.path.exists(f"{OUT}/training_args.bin"):
    os.remove(f"{OUT}/training_args.bin")
shutil.make_archive(OUT, "zip", ".", OUT)
from google.colab import files
files.download(f"{OUT}.zip")